# 多肢選択式の問題を解く

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/19_qa/multiple_choice_qa.ipynb)

NLP クックブック 19-06 ／ 質問応答

選択肢の中から正しい答えを 1 つ選ぶ形式の問題を解かせたい。埋め込みによる単純な方法と、LLM に選ばせる方法を比べたい。

In [1]:
!pip install -q datasets

In [2]:
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

ds = load_dataset("sbintuitions/JCommonsenseQA", split="validation").select(range(100))   # 常識を問う 5 択問題 (JGLUE)
ex = ds[0]
print(ex["question"], [ex[f"choice{i}"] for i in range(5)], "正解:", ex["label"])

# 1. 埋め込み: 質問と最も似ている選択肢を選ぶ
embedder = SentenceTransformer("intfloat/multilingual-e5-small")
q_emb = embedder.encode(["query: " + e["question"] for e in ds], normalize_embeddings=True)
c_emb = embedder.encode(["query: " + e[f"choice{i}"] for e in ds for i in range(5)], normalize_embeddings=True).reshape(len(ds), 5, -1)
pred_emb = np.einsum("nd,ncd->nc", q_emb, c_emb).argmax(axis=1)
gold = np.array(ds["label"])
print(f"偶然の正解率 0.200 / 埋め込みの類似度 {(pred_emb == gold).mean():.3f}")

電子機器で使用される最も主要な電子回路基板の事をなんと言う？ ['掲示板', 'パソコン', 'マザーボード', 'ハードディスク', 'まな板'] 正解: 2
偶然の正解率 0.200 / 埋め込みの類似度 0.600


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"   # 3B の日本語モデル (CPU で 1 問あたり 1〜2 秒)
tokenizer = AutoTokenizer.from_pretrained(name)
llm = AutoModelForCausalLM.from_pretrained(name).eval()
digit_ids = [tokenizer(str(i), add_special_tokens=False)["input_ids"][-1] for i in range(1, 6)]   # 「1」〜「5」のトークン

@torch.no_grad()
def choose(question, choices):
    """選択肢に番号を振り、次のトークンとして各番号が出る確率を比べる (生成はしないので 1 回の計算で済む)"""
    listing = "\n".join(f"{i}. {c}" for i, c in enumerate(choices, 1))
    prompt = f"次の質問の答えとして最も適切なものを選び、番号だけを答えてください。\n\n質問: {question}\n{listing}"
    text = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True, tokenize=False)
    ids = tokenizer(text, add_special_tokens=False, return_tensors="pt")["input_ids"]
    probs = llm(ids).logits[0, -1, digit_ids].softmax(-1)
    return int(probs.argmax()), probs.max().item()

results = [choose(e["question"], [e[f"choice{i}"] for i in range(5)]) for e in ds]
pred_llm, conf = np.array([r[0] for r in results]), np.array([r[1] for r in results])
print(f"LLM に番号を選ばせる {(pred_llm == gold).mean():.3f}")

LLM に番号を選ばせる 0.820


In [4]:
# 確信度の高い問題ほど正解しているか / 間違えた問題を見る
order = np.argsort(-conf)
print(f"確信度の上位 50 問の正解率 {(pred_llm == gold)[order[:50]].mean():.3f} / 下位 50 問 {(pred_llm == gold)[order[50:]].mean():.3f}")
for i in np.where(pred_llm != gold)[0][:3]:
    e = ds[int(i)]
    print(f"{e['question']} → 予測: {e[f'choice{pred_llm[i]}']} / 正解: {e[f'choice{gold[i]}']}")

確信度の上位 50 問の正解率 0.900 / 下位 50 問 0.740
地中にある一定の大きさの空間のこと？ → 予測: 山腹 / 正解: 洞窟
乗り物が関係する言葉はどれ？ → 予測: 靴を履く / 正解: 車から降りる
スピードを超過して警察に捕まることといえば？ → 予測: 危険運転 / 正解: 交通違反


- 選択肢に番号を振って、番号のトークンの確率を比べると、選択肢が長くても公平に比較でき、1 問につき 1 回の計算で済む。
- LLM は選択肢の位置に偏りを持つことがある (1 番を選びやすいなど)。選択肢の順序を入れ替えて複数回解かせ、多数決をとると安定する。
- 埋め込みの類似度は「質問と似た語」を選ぶだけなので、常識や推論が要る問題には弱い。ただし、偶然よりどれだけ上かを知るためのベースラインとして有用。
- 文書を読んで選ぶ形式 (読解問題) なら、プロンプトに文書を加える。知識が足りない問題は、検索した文書を付ける (RAG) と改善する。